# Assignment 1: Dynamic Programming

---

## Task 1) Edit Distances

Implement the [Hamming](https://en.wikipedia.org/wiki/Hamming_distance) and [Levenshtein](https://en.wikipedia.org/wiki/Levenshtein_distance) (edit) distances and compute them for the for the following word pairs.
It may help to compute them by hand first.

<img src = "./assets/97090.jpg" width="33%" /> <img src = "./assets/97222.jpg" width="33%" /> <img src = "./assets/97669.jpg" width="33%" />

Aside from computing the distance (which is a scalar), do the backtrace and compute the edit transcript (and thus alignment).

---

In [1]:
import operator
import numpy as np
from functools import reduce
from scipy.spatial import distance

In [2]:
WORD_PAIRS = [
    ("GCGTATGAGGCTAACGC", "GCTATGCGGCTATACGC"),
    ("kühler schrank", "schüler krank"),
    ("the longest", "longest day"),
    ("nicht ausgeloggt", "licht ausgenockt"),
    ("gurken schaben", "schurkengaben")
]

In [3]:
def hamming(s1: str, s2: str) -> int:
    """
    Compute the hamming distance between two strings.

    Arguments:
    s1: First string of word pair.
    s2: Second string of word pair.

    Returns the distance as int.
    """
    if len(s1) != len(s2):
        print("lengths do not match; using approximation")

    diffs = 0
    for i in range(0, min(len(s1), len(s2))):
        if s1[i] != s2[i]:
            diffs += 1

    # add the length difference as approximation
    return diffs + abs(len(s1) - len(s2))

In [4]:
for wordpair in WORD_PAIRS:
    dist = hamming(s1=wordpair[0], s2=wordpair[1])
    print("hamming('{}', '{}') = {}".format(
        wordpair[0], wordpair[1], dist
    ))

### EXPECTED
# hamming('GCGTATGAGGCTAACGC', 'GCTATGCGGCTATACGC') = 10
# hamming('kühler schrank', 'schüler krank') = 13
# hamming('the longest', 'longest day') = 11
# hamming('nicht ausgeloggt', 'licht ausgenockt') = 4
# hamming('gurken schaben', 'schurkengaben') = 14

hamming('GCGTATGAGGCTAACGC', 'GCTATGCGGCTATACGC') = 10
lengths do not match; using approximation
hamming('kühler schrank', 'schüler krank') = 13
hamming('the longest', 'longest day') = 11
hamming('nicht ausgeloggt', 'licht ausgenockt') = 4
lengths do not match; using approximation
hamming('gurken schaben', 'schurkengaben') = 14


In [5]:
def levenshtein(s1: str, s2: str, cost={'m': 0, 's': 1, 'i': 1, 'd': 1}) -> (int, str):
    """
    Compute the levenshtein (edit) distance between two strings.

    Arguments:
    s1: First string of word pair.
    s2: Second string of word pair.

    Returns the distance as int and edit transcript as string.
    """
    D = np.zeros((len(s1) + 1, len(s2) + 1), dtype=int)

    # for the empty word, costs match the length of the other string
    D[0, 1:] = range(1, len(s2) + 1)
    D[1:, 0] = range(1, len(s1) + 1)

    # this array will hold the journal of operations for backtracking
    T = np.zeros((len(s1) + 1, len(s2) + 1), dtype=np.object_)
    T[0, 0] = 'ε'
    T[0, 1:] = 'i'
    T[1:, 0] = 'd'
    
    for i in range(1, len(s1) + 1):
        for j in range(1, len(s2) + 1):
            diag = 'm' if s1[i-1] == s2[j-1] else 's'
            
            costs = [
                ('d', D[i-1, j] + cost['d']),
                ('i', D[i, j-1] + cost['i']),
                (diag, D[i-1, j-1] + cost[diag])
            ]
        
            op, c = min(costs, key=operator.itemgetter(1))
            D[i, j] = c
            T[i, j] = op
    
    # compute trace
    a, b = len(s1), len(s2)
    tr = []
    while a > 0 or b > 0:
        op = T[a, b]
        tr.append(op)
        if op == 'm' or op == 's':
            a -= 1
            b -= 1
        elif op == 'd':
            a -= 1
        elif op == 'i':
            b -= 1
        else:
            raise ValueError('Invalid operator: ' + str(op))
    
    return D[len(s1), len(s2)], reduce(operator.add, reversed(tr))

In [6]:
for wordpair in WORD_PAIRS:
    dist, transcript = levenshtein(s1=wordpair[0], s2=wordpair[1])
    print("levenshtein('{}', '{}') = {} ({})".format(
        wordpair[0], wordpair[1], dist, transcript
    ))

### EXPECTED
# levenshtein('GCGTATGAGGCTAACGC', 'GCTATGCGGCTATACGC') = 3 (mmdmmmmsmmmmmimmmm)
# levenshtein('kühler schrank', 'schüler krank') = 6 (ssmimmmmsddmmmm)
# levenshtein('the longest', 'longest day') = 8 (ddddmmmmmmmiiii)
# levenshtein('nicht ausgeloggt', 'licht ausgenockt') = 4 (smmmmmmmmmmsmssm)
# levenshtein('gurken schaben', 'schurkengaben') = 7 (siimmmmmsdddmmmm)

levenshtein('GCGTATGAGGCTAACGC', 'GCTATGCGGCTATACGC') = 3 (mmdmmmmsmmmmmimmmm)
levenshtein('kühler schrank', 'schüler krank') = 6 (ssmimmmmsddmmmm)
levenshtein('the longest', 'longest day') = 8 (ddddmmmmmmmiiii)
levenshtein('nicht ausgeloggt', 'licht ausgenockt') = 4 (smmmmmmmmmmsmssm)
levenshtein('gurken schaben', 'schurkengaben') = 7 (siimmmmmsdddmmmm)


---

## Task 2) Basic Spelling Correction using Edit Distance

For spelling correction, we will use prior knowledge, to put _some_ learning into our system.

The underlying idea is the _Noisy Channel Model_, that is: The user _intends_ to write a word `w`, but through some noise in the process, happens to type the word `x`.

The correct word $\hat{w}$ is that word, that is a valid candidate and has the highest probability:

$$
\begin{eqnarray}
\DeclareMathOperator*{\argmax}{argmax}
\hat{w} & = & \argmax_{w \in V} P(w | x) \\
        & = & \argmax_{w \in V} \frac{P(x|w) P(w)}{P(x)} \\
        & = & \argmax_{w \in V} P(x|w) P(w)
\end{eqnarray}
$$

1. The candidates $V$ can be obtained from a _vocabulary_.
2. The probability $P(w)$ of a word $w$ can be _learned (counted) from data_.
3. The probability $P(x|w)$ is more complicated... It could be learned from data, but we could also use a _heuristic_ that relates to the edit distance, e.g. rank by distance.

You may use [Peter Norvig's count_1w.txt](http://norvig.com/ngrams/) file, [local mirror](res/count_1w.tar.bz2).
Note that it may help to restrict to the first 10k words to get started.

---

In [7]:
EXAMPLES = [
    "pirates",    # in-voc
    "pirutes",    # pirates?
    "continoisly",  # continuosly?
]

In [8]:
### TODO: Prepare the vocabulary

# contains lines of "word <count>"
COUNTS_FILE = "data/count_1w.txt"

# read in vocabulary
voc = {}
all_counts = 0
num = 0

with open(COUNTS_FILE, "rb") as f:
    for line in f:
        w, c = line.strip().split()
        voc[w.decode('ascii')] = int(c)
        all_counts += int(c)
        num += 1

# normalize the counts
for k in voc:
    voc[k] = np.log(voc[k] / all_counts)
    
print("Read in {} lemmas.".format(len(voc)))

Read in 333333 lemmas.


In [9]:
def suggest(w: str, dist_fn, max_cand=5, max_dist=4) -> list:
    """
    Compute suggestions for spelling correction using edit distance.
    
    Arguments:
    w: Word in question.
    dist_fn: Distance function to use (e.g. levenshtein).
    max_cand: Maximum number of suggestions.

    Returns a list of tuples (word, dist, score) sorted by score and distance.
    """
    # if we have an exact hit, just return that.
    if w in voc:
        return [(w, 0, voc[w])]
    
    # maps a word to (other, edit-dist, other-rel-freq)
    def check(w, kv):
        ed = dist_fn(w, kv[0])
        return (kv[0], ed[0], kv[1])
    
    # compute edit distance of all words that differ at most max_dist in length
    res = [check(w, kv) for kv in voc.items() if abs(len(w) - len(kv[0])) < max_dist]
    
    # now sort descending by relative frequency then ascending by edit distance
    res = sorted(res, key=operator.itemgetter(2), reverse=True)
    res = sorted(res, key=operator.itemgetter(1))

    return res[:max_cand]

In [10]:
# How does your suggest function behave with levenshtein distance?

for word in EXAMPLES:
    suggestions = suggest(w=word, dist_fn=levenshtein, max_cand=3)
    print("{} {}".format(word, suggestions))

### EXPECTED
### Notice: your scores may vary!
# pirates [('pirates', 0, -11.408058827802126)]
# pirutes [('pirates', 1, -11.408058827802126), ('minutes', 2, -8.717825438953103), ('viruses', 2, -11.111468702571859)]
# continoisly [('continously', 1, -15.735337826575178), ('continuously', 2, -11.560071979871001), ('continuosly', 2, -17.009283000138204)]

pirates [('pirates', 0, -11.408058827802126)]
pirutes [('pirates', 1, -11.408058827802126), ('minutes', 2, -8.717825438953103), ('viruses', 2, -11.111468702571859)]
continoisly [('continously', 1, -15.735337826575178), ('continuously', 2, -11.560071979871001), ('continuosly', 2, -17.009283000138204)]


---

## Task 3) Needleman-Wunsch: Keyboard-aware Auto-Correct

In the parts 1 and 2 above, we applied uniform cost to all substitutions.
This does not really make sense if you look at a keyboard: the QWERTY layout will favor certain substitutions (eg. _a_ and _s_), while others are fairly unlikely (eg. _a_ and _k_).

Implement the [Needleman-Wunsch algorithm](https://en.wikipedia.org/wiki/Needleman–Wunsch_algorithm) which is very similar to the [Levenshtein distance](https://en.wikipedia.org/wiki/Levenshtein_distance), however it doesn't _minimize the cost_ but _maximizes the similarity_.
For a good measure of similarity, implement a metric that computes a meaningful weight for a given character substitution.

---

In [11]:
KEYBOARD = [
    'qwertyuiop[]\\',
    "asdfghjkl;'",
    ' zxcvbnm,./'
]

def get_c(c):
    for i, row in enumerate(KEYBOARD):
        if c in row:
            return i, row.index(c)
    raise ValueError(c + " not found on keyboad")

In [12]:
def keyboard_sim(s1: str, s2: str) -> float:
    # ensure to use similarity and not distance
    return 1 / (1 + distance.euclidean(get_c(s1), get_c(s2)))


def nw(s1: str, s2: str, d: float, sim_fn) -> float:
    """
    Apply needleman-wunsch algorithm.
    
    Arguments:
    s1: First string of word pair.
    s2: Second string of word pair.
    d: Gap penalty score.
    sim_fn: Similarity function to use.

    Returns the score as float.
    """
    D = np.zeros((len(s1) + 1, len(s2) + 1), dtype=int)

    D[0, 1:] = range(1, len(s2) + 1)
    D[0, 1:] *= d
    D[1:, 0] = range(1, len(s1) + 1)
    D[1:, 0] *= d

    for i in range(1, len(s1)+1):
        for j in range(1, len(s2)+1):
            match = D[i-1, j-1] + sim_fn(s1[i-1], s2[j-1])
            insert = D[i, j-1] + d
            delete = D[i-1, j] + d
            D[i, j] = max(match, insert, delete)
    # return negative distance (to use suggest function defined before)
    return D[len(s1)][len(s2)]


# make sure to use (-1)*nw since it maximizes similarity (ie, you need to reverse the score)
def nw_based_dist(s1: str, s2: str) -> (int, str):
    """
    Compute the needleman-wunsch distance between two strings.
    
    Arguments:
    s1: First string of word pair.
    s2: Second string of word pair.
    
    Returns the distance as int and <unsupported> string.
    """
    return -nw(s1, s2, -1, sim_fn=keyboard_sim), "<unsupported>"

In [13]:
# How does your suggest function behave with nw and a keyboard-aware similarity?

for word in EXAMPLES:
    suggestions = suggest(w=word, dist_fn=nw_based_dist, max_cand=3)
    print("{} {}".format(word, suggestions))

pirates [('pirates', 0, -11.408058827802126)]
pirutes [('pirates', -6, -11.408058827802126), ('minutes', -5, -8.717825438953103), ('viruses', -5, -11.111468702571859)]
continoisly [('continously', -10, -15.735337826575178), ('continuously', -9, -11.560071979871001), ('continuosly', -9, -17.009283000138204)]


### Efficient Implementation using Trie

In [14]:
class PrefixTree:    
    voc = dict()
    def __init__(self, parent, prefix, count=None):
        self.parent = parent
        self.prefix = prefix
        self.count = count
        self.succ = dict()
        self.hypref = None
    
    def size(self):
        return len(voc)
    
    def clean_hyprefs(self):
        agenda = [self]
        while agenda:
            n = agenda.pop()
            n.hypref = {}
            agenda.extend([s for (k, s) in n.succ.items()])    
    
    def insert(self, word, count):
        it = self
        for (i, c) in enumerate(word):
            if not c in it.succ:
                it.succ[c] = PrefixTree(it, word[:i+1])
            it = it.succ[c]
        if it.count:
            raise ValueError("{} already in tree".format(word))
        it.count = count
        self.voc[word] = count

    # query the score of a word in the tree
    def query(self, word):
        it = self
        for i in word:
            if not i in it.succ:
                raise ValueError("{} not found".format(word))
            it = it.succ[i]
        return it.count
    
    def __str__(self):
        return str({'prefix': self.prefix, 'succ': list(self.succ.keys()), 'count': self.count})
    
    def __repr__(self):
        return self.prefix if self.prefix else 'None'

    def to_string_lines(self):
        res = []
        agenda = [(n, 1) for (k, n) in sorted(self.succ.items(), reverse=True)]
        while agenda:
            n, d = agenda.pop()
            if not n.count:
                res.append("{} {}".format(' '*d, n.prefix))
            else:
                res.append("{} {} {}".format(' '*d, n.prefix, n.count))
                
            for (k, s) in sorted(n.succ.items(), reverse=True):
                agenda.append((s, d+1))
        
        return res

In [15]:
# read all entries into the prefix tree
root = PrefixTree(None, 'ε')

# reload the voc since we're now working with counts directly
with open(COUNTS_FILE, "rb") as f:
    for line in f:
        w, c = line.strip().split()
        root.insert(w.decode('ascii'), int(c))

print("Indexed {} words".format(len(root.voc.keys())))

Indexed 333333 words


In [16]:
# efficient implementation of edit distance for large vocabulary
def edit_with_trie(root, w, max_dist=3, cost={'m': 0, 's': 1, 'i': 1, 'd': 1}):
    # effectively, we'll build a shadow tree with refs to the original nodes 
    # initial, eps-row in D
    eps = {
        'token': 'ε',
        'noderef': root,
        'backref': None,
        'succ': dict(),
        'D': list(range(len(w)+1))
    }
    
    # populate the eps-cols; use items to be able to sort by key
    agenda = [(eps, 0)]
    while agenda:
        cur, depth = agenda.pop()
    
        for (c, n) in sorted(cur['noderef'].succ.items()):
            node = {
                'token': c,
                'noderef': n,
                'backref': cur,
                'succ': dict(),
                'D': [depth+1]
            }
            cur['succ'][c] = node
            agenda.append((node, depth+1))
    
    # we'll do a depth-first search, one char at a time
    eds = {}
    for (j, c) in enumerate(w, start=1):
        # start at depth=1
        agenda = [(t, sn, 1) for (t, sn) in sorted(eps['succ'].items())]
        while agenda:
            token, shadow_node, depth = agenda.pop()
            delta = cost['m'] if c == token else cost['s']
            
            # costs for each step
            cost_del = shadow_node['backref']['D'][j] + cost['d']  # D[i-1, j] one letter "up" = backref!
            cost_ins = shadow_node['D'][j-1] + cost['i']           # D[i, j-1] one letter "left" = same line
            cost_dia = shadow_node['backref']['D'][j-1] + delta    # D[i-1, j-1] one up+left

            # ...decide
            step = min(cost_del, cost_ins, cost_dia)
        
            shadow_node['D'].append(step)
            
            agenda.extend([(t, sn, depth+1) for (t, sn) in sorted(shadow_node['succ'].items())])                
            
            # at the end of the input word, if we have a word, update the edit distances accordingly
            if j == len(w) and step < max_dist:
                n = shadow_node['noderef']
                if n.count:
                    eds[n.prefix] = step  # (step, n.count)
            
    return eds

In [17]:
class Hyp:
    _cost = {'m': 0, 's': 1, 'i': 1, 'd': 1}
    def __init__(self, d, j, noderef):
        self.d = d  # depth (=row number)
        self.j = j  # character offset (=col number)
        self.noderef = noderef
        
        self.c = -1
        # some cost can be found right there
        if d == 0 and j == 0:
            self.c = 0
        elif d == 0:
            self.c = j * Hyp._cost['i']
        elif j == 0:
            self.c = d * Hyp._cost['d']
        
        # back-refs
        self.refi = None
        self.refd = None
        self.refs = None
    
    def __str__(self):
        return "({}, {}, {}, {})".format(self.noderef.prefix, self.d, self.j, self.c)
    
    def __eq__(self, other):
        return self.d == other.d and self.j == other.j and self.noderef == other.noderef

    def cost(self, word):
        if self.c < 0:
            vals = []
            if self.refs:
                delta = Hyp._cost['m'] if self.noderef.prefix[-1] == word[self.j-1] else Hyp._cost['s']
                vals.append(self.refs.cost(word) + delta)
            if self.refd:
                vals.append(self.refd.cost(word) + Hyp._cost['d'])
            if self.refi:
                vals.append(self.refi.cost(word) + Hyp._cost['i'])
            if not vals:
                raise ValueError("Unable to obtain cost; check agenda sorting! %s %s" % (str(self), word))

            self.c = min(vals)

        return self.c

In [18]:
def edit_with_hypothesis(root, w, max_cost=3):
    # clear out any hypref in the tree
    root.clean_hyprefs()
    
    # edit distances to return
    eds = {}
    
    # initial hypothesis
    eps = Hyp(0, 0, root)
    root.hypref[0] = eps
    
    agenda = [eps]
    while agenda:
        h = agenda.pop(0)
        
        if h.cost(w) > max_cost:
            continue
            
        if h.j == len(w):
            n = h.noderef
            if n.count:
                eds[n.prefix] = h.cost(w)
        
        # don't expand longer than the word
        if h.j in h.noderef.hypref:
            i = h.noderef.hypref[h.j]
            i.refi = h
        else:
            if h.j < len(w) and h.j * Hyp._cost['i'] < max_cost:
                i = Hyp(h.d, h.j+1, h.noderef)
                i.refi = h
                if h.refd:
                    i.refs = h.refd
                agenda.append(i)
                h.noderef.hypref[h.j] = i
        
        # only expand "downwards" if there's more successors (depth)
        for (t, n) in h.noderef.succ.items():
            if h.j in n.hypref:
                d = n.hypref[h.j]
                d.refd = h
            else:
                if h.d * Hyp._cost['d'] < max_cost:
                    d = Hyp(h.d+1, h.j, n)
                    d.refd = h
                    if h.refi:
                        d.refs = h.refi
                    agenda.append(d)
                    n.hypref[h.j] = d
        
            if (h.j+1) in n.hypref:
                s = n.hypref[h.j+1]
                s.refs = h
            else:
                if h.j < len(w):
                    s = Hyp(h.d+1, h.j+1, n)
                    s.refs = h
                    agenda.append(s)
                    n.hypref[h.j+1] = s
    
    return eds

In [19]:
# compare timings
for e in EXAMPLES[0:1]:
    print(e)
    
    print("\nClassical implementation")
    %time individual = {w: levenshtein(e, w)[0] for w in root.voc}
    print(sorted(individual.items(), key=operator.itemgetter(1))[:10])
    
    print("\nPrefix tree implementation")
    %time prefixed = edit_with_trie(root, e)
    print(sorted(prefixed.items(), key=operator.itemgetter(1))[:10])

    print("\nHypothesis implementation")
    %time hypothesis = edit_with_hypothesis(root, e)
    print(sorted(hypothesis.items(), key=operator.itemgetter(1))[:10])

pirates

Classical implementation
CPU times: user 17.8 s, sys: 13.6 ms, total: 17.8 s
Wall time: 17.9 s
[('pirates', 0), ('pirate', 1), ('pilates', 1), ('pirated', 1), ('piratas', 1), ('pyrates', 1), ('rates', 2), ('plates', 2), ('operates', 2), ('emirates', 2)]

Prefix tree implementation
CPU times: user 5.47 s, sys: 53.8 ms, total: 5.52 s
Wall time: 5.54 s
[('pirates', 0), ('pyrates', 1), ('pirate', 1), ('pirated', 1), ('piratas', 1), ('pilates', 1), ('vibrates', 2), ('trates', 2), ('tirades', 2), ('timates', 2)]

Hypothesis implementation
CPU times: user 11.2 s, sys: 37.4 ms, total: 11.3 s
Wall time: 11.3 s
[('pirates', 0), ('pilates', 1), ('pirated', 1), ('piratas', 1), ('pyrates', 1), ('timates', 2), ('tirades', 2), ('berates', 2), ('borates', 2), ('mirages', 2)]
